<a href="https://colab.research.google.com/github/Srivighnesh/jekins-test/blob/main/socailMedia_DBSCAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.manifold import TSNE # Import t-SNE
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

In [6]:
path = kagglehub.dataset_download("hamnamunir/social-media-user-behavior-dataset")

Using Colab cache for faster access to the 'social-media-user-behavior-dataset' dataset.


In [7]:
df = pd.read_csv('/kaggle/input/social-media-user-behavior-dataset/social_media_user_behavior.csv')

In [8]:
df['account_join_date'] = pd.to_datetime(df['account_join_date'])
df['join_day'] = df['account_join_date'].dt.day
df['join_month'] = df['account_join_date'].dt.month
df['join_year'] = df['account_join_date'].dt.year

print("New columns created from 'account_join_date':")
print(df[['account_join_date', 'join_day', 'join_month', 'join_year']].head())

New columns created from 'account_join_date':
  account_join_date  join_day  join_month  join_year
0        2022-10-21        21          10       2022
1        2024-12-03         3          12       2024
2        2023-03-29        29           3       2023
3        2024-04-26        26           4       2024
4        2023-09-08         8           9       2023


In [9]:
drop_cols = [
    'user_id', # Removing it bcs doesn't describe user behaviour
    'account_join_date', # we have already created separate date, year, month columns
]
df_cleaned = df.drop(columns=[col for col in drop_cols if col in df.columns])

In [ ]:
# Separate numeric and categorical columns

numeric_features = df_cleaned.select_dtypes(include=['number']).columns.tolist()
categorical_features = df_cleaned.select_dtypes(include=['object', 'category']).columns.tolist()

# Remove old cluster column if it already exists
if 'kmeans_cluster' in numeric_features:
    numeric_features.remove('kmeans_cluster')

if 'kmeans_cluster' in categorical_features:
    categorical_features.remove('kmeans_cluster')

print("Numeric Features:")
print(numeric_features)
print("\nCategorical Features:")
print(categorical_features)

Numeric Features:
['age', 'platforms_used_count', 'daily_usage_hours', 'sessions_per_day', 'avg_session_duration_min', 'followers_count', 'following_count', 'posts_per_week', 'likes_given_per_day', 'comments_per_day', 'shares_per_day', 'dms_sent_per_day', 'ad_click_rate', 'monthly_spend_via_social_usd', 'self_reported_mental_health_score', 'join_day', 'join_month', 'join_year']

Categorical Features:
['gender', 'country', 'profession', 'primary_platform', 'preferred_device', 'peak_usage_time', 'preferred_content_type', 'scroll_speed', 'purchased_via_social_media', 'primary_purpose', 'sleep_disruption', 'screen_time_concern', 'notification_frequency', 'privacy_setting', 'influencer_status', 'mood_while_scrolling', 'takes_social_media_breaks']


In [ ]:
# combine selected features
features = numeric_features + categorical_features
# create copy for clustering
data = df[features].copy()
data.head()

,age,platforms_used_count,daily_usage_hours,sessions_per_day,avg_session_duration_min,followers_count,following_count,posts_per_week,likes_given_per_day,comments_per_day,...,scroll_speed,purchased_via_social_media,primary_purpose,sleep_disruption,screen_time_concern,notification_frequency,privacy_setting,influencer_status,mood_while_scrolling,takes_social_media_breaks
0,30,1,1.6,2,48.0,353,99,1,17,1,...,Medium,Yes,Entertainment,No impact,No,Do Not Disturb,Friends Only,No,Neutral,No
1,25,2,2.4,5,28.8,276,514,2,28,3,...,Medium,Yes,News & Updates,Moderate impact,No,Selected,Public,No,Neutral,Yes
2,32,5,0.5,6,5.0,203,422,3,26,4,...,Slow,No,Learning,Mild impact,Yes,Always On,Friends Only,No,Happy,Yes
3,39,4,3.0,7,25.7,29083,474,1,16,3,...,Fast,No,Entertainment,Mild impact,Somewhat,Always On,Public,Yes,Happy,No
4,25,3,3.9,7,33.4,104,197,2,27,7,...,Fast,No,Entertainment,Mild impact,No,Do Not Disturb,Public,No,Inspired,No


In [ ]:
# encoding
x = pd.get_dummies(df_cleaned, drop_first=True)
X_scaled = StandardScaler().fit_transform(x)

In [ ]:

# PCA for visualization before Applying DBSCAN
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Create a DataFrame for Plotly for easier labeling
pca_df = pd.DataFrame(data=X_pca, columns=['Principal Component 1', 'Principal Component 2'])

# Create the interactive scatter plot
fig = px.scatter(pca_df,
                 x='Principal Component 1',
                 y='Principal Component 2',
                 title='Interactive Scatter Plot of Data (First Two Principal Components)',
                 width=800,
                 height=500,
                 opacity=0.7,
                 color_discrete_sequence=['deepskyblue'], # Set a color for the points
                 hover_data={'Principal Component 1': ':.2f', 'Principal Component 2': ':.2f'}) # Show coordinates on hover

fig.update_layout(title_font_size=18, xaxis_title_font_size=14, yaxis_title_font_size=14)
fig.update_traces(marker=dict(size=8, line=dict(width=1, color='DarkSlateGrey')))

fig.show()

# DBSCAN Clustering

In [ ]:
# Choosing Correct EPS value
for eps in [9, 10, 11, 12, 13, 15]:
    labels = DBSCAN(
        eps=eps,
        min_samples=5
    ).fit_predict(X_scaled)

    cluster_counts = pd.Series(labels).value_counts().sort_index()

    print(f"\neps={eps}")
    print(cluster_counts)


eps=9
-1    1589
 0     384
 1       4
 2       5
 3       5
 4       5
 5       2
 6       5
 7       1
Name: count, dtype: int64

eps=10
-1     416
 0    1576
 1       3
 2       5
Name: count, dtype: int64

eps=11
-1      60
 0    1940
Name: count, dtype: int64

eps=12
-1       6
 0    1994
Name: count, dtype: int64

eps=13
-1       1
 0    1999
Name: count, dtype: int64

eps=15
-1       1
 0    1999
Name: count, dtype: int64


In [ ]:
# DBSCAN clustering
dbscan = DBSCAN(eps=10, min_samples=6)
labels = dbscan.fit_predict(X_scaled)

In [ ]:
# t-SNE for 2D visualization
# Note: t-SNE can be computationally intensive for very large datasets.
# A random state is used for reproducibility.
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=300) # Added perplexity and n_iter for better results
X_tsne = tsne.fit_transform(X_scaled)

# Create DataFrame
tsne_df = pd.DataFrame(
    X_tsne,
    columns=[
        't-SNE Component 1',
        't-SNE Component 2' # Only two components
    ]
)

tsne_df['Cluster'] = labels.astype(str)

# Interactive 2D scatter plot
fig = px.scatter(
    tsne_df, # Use tsne_df
    x='t-SNE Component 1',
    y='t-SNE Component 2',
    color='Cluster',
    title='2D t-SNE Visualization of DBSCAN Clusters',
    width=900,
    height=600,
    opacity=0.7,
    hover_data={
        't-SNE Component 1': ':.2f',
        't-SNE Component 2': ':.2f',
        'Cluster': True
    }
)

fig.update_traces(
    marker=dict(
        size=8,
        line=dict(width=0.5, color='gainsboro')
    )
)

fig.update_layout(
    title_font_size=18,
    xaxis_title='t-SNE 1', # Updated axis titles
    yaxis_title='t-SNE 2'
)

fig.show()

/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning:

'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.



In [ ]:
cluster_size = pd.Series(labels).value_counts()
cluster_size # Count Cluster Count

,count
0,1552
-1,448


In [ ]:
df_cleaned['Cluster'] = labels
outliers = df_cleaned[
    df_cleaned['Cluster'] == 0
]

print(outliers.shape)

outliers.sort_values(
    'followers_count',
    ascending=False
).head(10)

(1552, 36)


,age,gender,country,profession,primary_platform,platforms_used_count,daily_usage_hours,sessions_per_day,avg_session_duration_min,preferred_device,...,screen_time_concern,notification_frequency,privacy_setting,influencer_status,mood_while_scrolling,takes_social_media_breaks,join_day,join_month,join_year,Cluster
1793,32,Female,UK,Student,LinkedIn,4,1.2,6,12.0,Smartphone,...,Somewhat,Always On,Public,Yes,Relaxed,No,25,8,2023,0
371,30,Female,USA,Freelancer,YouTube,2,0.5,4,7.5,Smartphone,...,Yes,Do Not Disturb,Friends Only,Yes,Happy,Yes,28,2,2023,0
857,30,Male,Pakistan,Student,Twitter/X,4,1.5,3,30.0,Smartphone,...,Yes,Selected,Private,Yes,Happy,No,27,6,2023,0
123,15,Male,Australia,Researcher,Twitter/X,4,2.3,6,23.0,Smartphone,...,Yes,Do Not Disturb,Public,Yes,Neutral,Occasionally,16,9,2024,0
1821,27,Male,Pakistan,Other,Twitter/X,2,2.2,10,13.2,Smartphone,...,Yes,Selected,Public,Yes,Relaxed,Occasionally,18,11,2024,0
1750,13,Male,UK,Freelancer,TikTok,5,2.3,3,46.0,Smartphone,...,Somewhat,Selected,Public,Yes,Bored,Yes,30,12,2022,0
1360,26,Female,Pakistan,Doctor,Instagram,2,1.9,6,19.0,Smartphone,...,No,Always On,Public,Yes,Neutral,Yes,26,8,2023,0
989,27,Male,USA,Teacher,Instagram,1,2.1,7,18.0,Smartphone,...,No,Off,Public,Yes,Relaxed,No,17,2,2023,0
58,29,Female,India,Freelancer,Instagram,3,3.8,6,38.0,Desktop,...,Somewhat,Do Not Disturb,Public,Yes,Neutral,Occasionally,12,5,2023,0
1987,31,Female,Pakistan,Student,TikTok,1,3.3,7,28.3,Smartphone,...,No,Selected,Public,Yes,Happy,Occasionally,24,3,2022,0


Mapping clusters by names

In [ ]:
cluster_names = {
    -1: "High Visibility Users(outliners)",
     0: "Gernal Users",
}

df_cleaned["Cluster_Name"] = df_cleaned["Cluster"].map(cluster_names)

In [ ]:
# Percentage crosstab of features
for feature in features:
  if feature in categorical_features:
    print(f"\n{feature} is categorical feature")
    display(
        pd.crosstab(
            index=df_cleaned['Cluster_Name'],
            columns=df_cleaned[feature],
            normalize='index'
        ).round(2) * 100
    )
  else:
    print("---------------------------------")
    print(f"\n{feature} in numerical feature")
    # For numerical features, crosstab is generally not appropriate for distribution
    # as it treats each unique numerical value as a category. If a summary statistic
    # (like mean) per cluster is desired, df_cleaned.groupby('Cluster_Name')[feature].mean()
    # would be more appropriate. For now, we keep the original structure.
    display(
        pd.crosstab(
            df_cleaned['Cluster_Name'],
            df_cleaned[feature],
            normalize='index'
        ).round(2)
    )

---------------------------------

age in numerical feature


age,13,14,15,16,17,18,19,20,21,22,...,42,43,44,45,46,47,48,51,52,57
Cluster_Name,,,,,,,,,,,,,,,,,,,,,
Gernal Users,0.04,0.02,0.02,0.01,0.02,0.03,0.03,0.03,0.04,0.04,...,0.01,0.01,0.01,0.0,0.0,0.0,0.0,0.0,0.0,0.0
High Visibility Users(outliners),0.05,0.02,0.02,0.02,0.01,0.03,0.03,0.04,0.03,0.08,...,0.00,0.01,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0


---------------------------------

platforms_used_count in numerical feature


platforms_used_count,1,2,3,4,5
Cluster_Name,,,,,
Gernal Users,0.20,0.19,0.21,0.18,0.21
High Visibility Users(outliners),0.21,0.18,0.20,0.19,0.22


---------------------------------

daily_usage_hours in numerical feature


...

influencer_status,No,Yes
Cluster_Name,,
Gernal Users,96.0,4.0
High Visibility Users(outliners),83.0,17.0



mood_while_scrolling is categorical feature


mood_while_scrolling,Bored,Happy,Inspired,Neutral,Relaxed,Stressed
Cluster_Name,,,,,,
Gernal Users,17.0,18.0,17.0,15.0,17.0,16.0
High Visibility Users(outliners),18.0,15.0,18.0,18.0,16.0,14.0



takes_social_media_breaks is categorical feature


takes_social_media_breaks,No,Occasionally,Yes
Cluster_Name,,,
Gernal Users,32.0,34.0,34.0
High Visibility Users(outliners),27.0,37.0,36.0


Cluster Profling

In [ ]:
# Iterate through each categorical feature and create a crosstab
for feature in categorical_features:
    print(f"\nDistribution of '{feature.replace('_', ' ').title()}' across Clusters:")

    # Generate crosstab, normalizing by index to get percentages within each cluster
    cross_tab = pd.crosstab(
        df_cleaned['Cluster_Name'],
        df_cleaned[feature],
        normalize='index'
    ) * 100 # Convert to percentage

    display(cross_tab.round(2)) # Display with 2 decimal places


Distribution of 'Gender' across Clusters:


gender,Female,Male,Non-binary,Prefer not to say
Cluster_Name,,,,
Gernal Users,51.03,48.39,0.52,0.06
High Visibility Users(outliners),39.96,38.17,12.95,8.93



Distribution of 'Country' across Clusters:


country,Australia,Brazil,Canada,Germany,India,Nigeria,Pakistan,UAE,UK,USA
Cluster_Name,,,,,,,,,,
Gernal Users,7.86,3.35,7.09,6.31,16.04,3.87,20.75,8.96,8.57,17.20
High Visibility Users(outliners),4.91,8.71,11.83,8.93,16.52,8.26,10.49,9.60,10.27,10.49



Distribution of 'Profession' across Clusters:


...

influencer_status,No,Yes
Cluster_Name,,
Gernal Users,96.01,3.99
High Visibility Users(outliners),82.59,17.41



Distribution of 'Mood While Scrolling' across Clusters:


mood_while_scrolling,Bored,Happy,Inspired,Neutral,Relaxed,Stressed
Cluster_Name,,,,,,
Gernal Users,17.14,18.30,16.69,15.46,16.82,15.59
High Visibility Users(outliners),18.30,14.96,18.08,18.30,16.07,14.29



Distribution of 'Takes Social Media Breaks' across Clusters:


takes_social_media_breaks,No,Occasionally,Yes
Cluster_Name,,,
Gernal Users,31.70,34.34,33.96
High Visibility Users(outliners),27.01,36.83,36.16


# Find pattens Acording to categries

### feature display functions

In [ ]:

def compare_two_features_with_clusters(df, feature1_name, feature2_name):
    """
    Compares the distribution of two features across different clusters using a scatter plot.
    This function is primarily designed for numerical features, displaying clusters via color.

    Args:
        df (pd.DataFrame): The input DataFrame containing the features and 'Cluster_Name'.
        feature1_name (str): The name of the first feature (for x-axis).
        feature2_name (str): The name of the second feature (for y-axis).
    """
    if feature1_name not in numeric_features or feature2_name not in numeric_features:
        print(f"Error: Both '{feature1_name}' and '{feature2_name}' must be numerical features "
              "for this type of comparison. Please choose two numerical features.")
        # You could extend this to handle categorical features with different plot types (e.g., grouped bar charts, heatmaps)
        return

    print(f"--- Comparing {feature1_name.replace('_', ' ').title()} vs {feature2_name.replace('_', ' ').title()} by Cluster ---")

    fig = px.scatter(
        df,
        x=feature1_name,
        y=feature2_name,
        color='Cluster_Name',
        title=f'Comparison of {feature1_name.replace("_", " ").title()} and {feature2_name.replace("_", " ").title()} by Cluster',
        hover_name='Cluster_Name',
        width=900,
        height=600,
        color_discrete_sequence=px.colors.qualitative.Set2
    )

    fig.update_layout(
        xaxis_title=feature1_name.replace("_", " ").title(),
        yaxis_title=feature2_name.replace("_", " ").title(),
        title_font_size=18,
        legend_title_text='Cluster'
    )

    fig.show()

In [ ]:
def features_visuals(df_cleaned, engagement_bins):
    engagement_bins = engagement_bins

    for feature, config in engagement_bins.items():

        temp_col = f"{feature}_group"

        # Create ordered categorical type
        binned_type = pd.CategoricalDtype(
            categories=config['labels'],
            ordered=True
        )

        df_cleaned[temp_col] = pd.cut(
            df_cleaned[feature],
            bins=config['bins'],
            labels=config['labels'],
            include_lowest=True
        ).astype(binned_type)

        # Generate charts for each cluster
        for cluster_name in df_cleaned['Cluster_Name'].unique():

            cluster_data = df_cleaned[
                df_cleaned['Cluster_Name'] == cluster_name
            ]

            # Calculate raw counts instead of percentages
            count_data = (
                cluster_data[temp_col]
                .value_counts()
                .reindex(config['labels'])
                .fillna(0)
                .astype(int)
                .reset_index()
            )

            count_data.columns = [temp_col, 'Count']

            fig = px.bar(
                count_data,
                x=temp_col,
                y='Count',
                color=temp_col,
                title=f"{feature.replace('_', ' ').title()} Distribution for {cluster_name}",
                width=800,
                height=500,
                color_discrete_sequence=px.colors.qualitative.Set2,
                text='Count' # Display raw count as text
            )

            fig.update_traces(
                texttemplate='%{text}', # Show raw number
                textposition='outside' # Keep outside for readability with numbers
            )

            fig.update_layout(
                xaxis_title=feature.replace("_", " ").title(),
                yaxis_title='Number of People', # Changed y-axis title
                legend_title=feature.replace("_", " ").title(),
                xaxis=dict(
                    categoryorder='array',
                    categoryarray=config['labels']
                ),
                showlegend=False
            )

            fig.show()

        # Remove temporary column
        df_cleaned.drop(columns=[temp_col], inplace=True)

    return None

In [ ]:
def features_categorical_socail(features, featureName):
    for f in features:

        fig = px.histogram(
            df_cleaned,
            x='Cluster_Name',
            color=f,width=800,height=500,
            barmode='group',  # similar to seaborn countplot
            title=f'{f.replace("_", " ").title()}',
            color_discrete_sequence=px.colors.qualitative.Set2
        )

        fig.update_layout(
            xaxis_title=featureName,
            yaxis_title='Number of Users',
            legend_title=f.replace("_", " ").title(),
            title_x=0.5,
            legend=dict(
                orientation="h",
                yanchor="bottom",
                title=None, # Removed title as requested
                y=-0.3, # Adjusted this value to increase margin below the plot
                xanchor="right",
                x=1
            ),
            margin=dict(t=50) # Added top margin
        )

        fig.show()

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

def smart_compare_features(df, feature_names):
    """
    Intelligently compares two features against clusters using suitable Plotly visualizations.

    Args:
        df (pd.DataFrame): The input DataFrame containing features and 'Cluster_Name'.
        feature_names (list): A list containing exactly two feature names to compare.
    """
    if not isinstance(feature_names, list) or len(feature_names) != 2:
        print("Error: 'feature_names' must be a list containing exactly two feature names.")
        return

    # Helper to check if a feature exists in our global lists
    def is_valid_feature(f_name):
        return f_name in numeric_features or f_name in categorical_features

    feature1, feature2 = feature_names[0], feature_names[1]

    if not is_valid_feature(feature1) or not is_valid_feature(feature2):
        print("Error: One or both features not recognized. Ensure they are in 'numeric_features' or 'categorical_features'.")
        return

    is_f1_numeric = feature1 in numeric_features
    is_f2_numeric = feature2 in numeric_features
    is_f1_categorical = feature1 in categorical_features
    is_f2_categorical = feature2 in categorical_features

    if is_f1_numeric and is_f2_numeric:
        # Check if 'followers_count' is one of the features and should be binned
        if 'followers_count' in [feature1, feature2] and 'followers_count' in engagement_bins:
            print(f"--- Comparing {feature1.replace('_', ' ').title()} vs {feature2.replace('_', ' ').title()} by Cluster (Numerical-Categorical Box Plot with Binned Followers Count) ---")

            temp_df = df.copy()
            binned_feature_name = ''
            numerical_feature_name = ''

            # Identify which feature is 'followers_count' and which is the other numerical feature
            if feature1 == 'followers_count':
                binned_feature_name = feature1
                numerical_feature_name = feature2
            else: # feature2 == 'followers_count'
                binned_feature_name = feature2
                numerical_feature_name = feature1

            config = engagement_bins[binned_feature_name]
            binned_col_name = f"{binned_feature_name}_binned"

            temp_df[binned_col_name] = pd.cut(
                temp_df[binned_feature_name],
                bins=config['bins'],
                labels=config['labels'],
                include_lowest=True
            ).astype(pd.CategoricalDtype(categories=config['labels'], ordered=True))

            fig = px.box(
                temp_df,
                x=binned_col_name,
                y=numerical_feature_name,
                color='Cluster_Name',
                title=f'Distribution of {numerical_feature_name.replace("_", " ").title()} by {binned_feature_name.replace("_", " ").title()} across Clusters',
                width=1000,
                height=600,
                color_discrete_sequence=px.colors.qualitative.Set2,
                points='outliers' # Show outlier points
            )
            fig.update_layout(
                xaxis_title=binned_feature_name.replace("_", " ").title() + ' (Binned)',
                yaxis_title=numerical_feature_name.replace("_", " ").title(),
                title_font_size=18,
                legend_title_text='Cluster'
            )
            fig.show()
            return # Exit after plotting the binned version

        # Default numerical-numerical scatter plot logic if no specific binning applied
        print(f"--- Comparing {feature1.replace('_', ' ').title()} vs {feature2.replace('_', ' ').title()} by Cluster (Numerical-Numerical Scatter Plot) ---")
        fig = px.scatter(
            df,
            x=feature1,
            y=feature2,
            color='Cluster_Name',
            title=f'Comparison of {feature1.replace("_", " ").title()} and {feature2.replace("_", " ").title()} by Cluster',
            hover_name='Cluster_Name',
            width=900,
            height=600,
            color_discrete_sequence=px.colors.qualitative.Set2
        )
        fig.update_layout(
            xaxis_title=feature1.replace("_", " ").title(),
            yaxis_title=feature2.replace("_", " ").title(),
            title_font_size=18,
            legend_title_text='Cluster'
        )
        fig.show()

    elif (is_f1_numeric and is_f2_categorical) or (is_f1_categorical and is_f2_numeric):
        print(f"--- Comparing {feature1.replace('_', ' ').title()} vs {feature2.replace('_', ' ').title()} by Cluster (Numerical-Categorical) ---")

        # Determine which feature is numerical and which is categorical
        if is_f1_numeric:
            numerical_feat = feature1
            categorical_feat = feature2
        else:
            numerical_feat = feature2
            categorical_feat = feature1

        fig = px.box(
            df,
            x=categorical_feat,
            y=numerical_feat,
            color='Cluster_Name',
            title=f'Distribution of {numerical_feat.replace("_", " ").title()} by {categorical_feat.replace("_", " ").title()} across Clusters',
            width=1000,
            height=600,
            color_discrete_sequence=px.colors.qualitative.Set2,
            points='outliers' # Show outlier points
        )
        fig.update_layout(
            xaxis_title=categorical_feat.replace("_", " ").title(),
            yaxis_title=numerical_feat.replace("_", " ").title(),
            title_font_size=18,
            legend_title_text='Cluster'
        )
        fig.show()

    else: # Both categorical
        print(f"--- Comparing {feature1.replace('_', ' ').title()} vs {feature2.replace('_', ' ').title()} by Cluster (Categorical-Categorical) ---")
        fig = px.histogram(
            df,
            x=feature1,
            color=feature2,
            facet_col='Cluster_Name',
            barmode='group',
            title=f'Distribution of {feature1.replace("_", " ").title()} by {feature2.replace("_", " ").title()} across Clusters',
            width=1200,
            height=600,
            color_discrete_sequence=px.colors.qualitative.Set2
        )
        fig.update_layout(
            xaxis_title=feature1.replace("_", " ").title(),
            yaxis_title='Count',
            legend_title_text=feature2.replace("_", " ").title(),
            title_font_size=18
        )
        fig.show()

In [ ]:
smart_compare_features(df_cleaned, ['primary_platform', 'influencer_status'])

--- Comparing Primary Platform vs Influencer Status by Cluster (Categorical-Categorical) ---


### User Behaviour features

In [ ]:
User_Behavior_features = [
    'daily_usage_hours',
    'sessions_per_day',
    'avg_session_duration_min',
    'scroll_speed',
    'notification_frequency',
    'takes_social_media_breaks',
    'peak_usage_time'
]
user_behavior_numeric_features = [f for f in User_Behavior_features if f in numeric_features]
user_behavior_categorical_features = [f for f in User_Behavior_features if f in categorical_features]

In [ ]:
behavior_profile = (
    df_cleaned
    .groupby('Cluster_Name')[User_Behavior_features]
    .mean(numeric_only=True)
    .round(2)
)
print(behavior_profile)

                                  daily_usage_hours  sessions_per_day  \
Cluster_Name                                                            
Gernal Users                                   2.90              5.07   
High Visibility Users(outliners)               3.31              4.85   

                                  avg_session_duration_min  
Cluster_Name                                                
Gernal Users                                         42.70  
High Visibility Users(outliners)                     64.34  


Compare Against Typical Users (Typical Users)

In [ ]:
features_categorical_socail(  user_behavior_categorical_features, 'User Behavior')

In [ ]:
import plotly.express as px
import pandas as pd

behavior_bins = {
    'daily_usage_hours': {
        'bins': [0, 1, 3, 6, 24],
        'labels': ['<1 hour', '1-3 hours', '3-6 hours', '>6 hours']
    },

    'sessions_per_day': {
        'bins': [0, 3, 6, 10, 50],
        'labels': ['<3', '3-6', '6-10', '10-50']
    },

    'avg_session_duration_min': {
        'bins': [0, 20, 45, 75, 300],
        'labels': ['<20 mins', '20-45 mins', '45-75 mins', '>75 mins']
    }
}


for feature, config in behavior_bins.items():

    temp_col = f"{feature}_group"

    df_cleaned[temp_col] = pd.cut(
        df_cleaned[feature],
        bins=config['bins'],
        labels=config['labels'],
        include_lowest=True
    )

    # Get unique cluster names to iterate through for individual pie charts
    for cluster_name in df_cleaned['Cluster_Name'].unique():
        # Filter data for the current cluster
        cluster_data = df_cleaned[df_cleaned['Cluster_Name'] == cluster_name]

        # Calculate proportions for the pie chart
        pie_data = cluster_data[temp_col].value_counts().reset_index()
        pie_data.columns = [temp_col, 'Count']

        fig = px.pie(
            pie_data,
            values='Count',
            names=temp_col, width=800,height=500,
            title=f'{feature.replace("_"," ").title()} Distribution for {cluster_name}',
            color=temp_col, # Use the group for color coding
            color_discrete_sequence=px.colors.qualitative.Set2
        )

        fig.update_traces(textposition='outside', textinfo='percent+label')
        fig.update_layout(
            legend_title=feature.replace("_"," ").title(),
        )

        fig.show()

    # Drop the temporary column after all plots for this feature are generated
    df_cleaned.drop(columns=[temp_col], inplace=True)

In [ ]:
fig = px.box(
    df_cleaned,
    x='Cluster_Name',
    y='daily_usage_hours',
    title='Daily Usage Hours by Cluster',
    width=800,
    height=500,
    color='Cluster_Name',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.show()

### Engagement features

In [ ]:
Engagement_features = np.array([
    'posts_per_week',
    'likes_given_per_day',
    'comments_per_day',
    'shares_per_day',
    'ad_click_rate',
    'preferred_content_type',
    'mood_while_scrolling'
])
engagement_numirical_features = [f for f in Engagement_features if f in numeric_features]
engagement_categorical_features = [f for f in Engagement_features if f in categorical_features]
for f in engagement_numirical_features:
    print(f)

posts_per_week
likes_given_per_day
comments_per_day
shares_per_day
ad_click_rate


In [ ]:
behavior_profile = (
    df_cleaned
    .groupby('Cluster_Name')[Engagement_features]
    .mean(numeric_only=True)
    .round(2)
)
print(behavior_profile)

                                  posts_per_week  likes_given_per_day  \
Cluster_Name                                                            
Gernal Users                                2.92                19.88   
High Visibility Users(outliners)            3.12                19.91   

                                  comments_per_day  shares_per_day  \
Cluster_Name                                                         
Gernal Users                                  3.96            1.97   
High Visibility Users(outliners)              4.10            2.07   

                                  ad_click_rate  
Cluster_Name                                     
Gernal Users                               0.13  
High Visibility Users(outliners)           0.14  


In [ ]:
engagement_bins_engagement = {
    'posts_per_week': {
        'bins': [0, 2, 4, 6, 10],
        'labels': ['Low Activity', 'Moderate Activity', 'Active', 'Highly Active']
    },

    'comments_per_day': {
        'bins': [0, 2, 4, 6, 11],
        'labels': ['Low Engagement', 'Moderate Engagement', 'Active', 'Highly Engaged']
    },

    'shares_per_day': {
        'bins': [0, 1, 3, 5, 8],
        'labels': ['Low Sharing', 'Moderate Sharing', 'Active Sharing', 'High Sharing']
    }
}

features_visuals(df_cleaned, engagement_bins_engagement) # Numirical features

In [ ]:
features_categorical_socail(engagement_categorical_features, 'Engagement') # Catagorical

In [ ]:
pd.crosstab(
    df_cleaned['Cluster_Name'],
    df_cleaned['age'],
    normalize='index'
).round(2) * 100

# Social Influence Features


In [ ]:
social_influence_features = np.array(
    [
        'followers_count',
        'following_count',
        'posts_per_week',
        'influencer_status',
        'primary_platform',
    ]
)
social_numirical_features = [f for f in social_influence_features if f in numeric_features]
social_categorical_features = [f for f in social_influence_features if f in categorical_features]
for f in social_numirical_features:
    print(f)

print('\n')
for f in social_categorical_features:
    print(f)

followers_count
following_count
posts_per_week


influencer_status
primary_platform


In [ ]:

engagement_bins = {
    'followers_count': {
        'bins': [0, 1000, 5000, 10000, 20000, np.inf],
        'labels': ['<1K', '1K-5K', '5K-10K', '10K-20K', '>20K']
    },

    'following_count': {
        'bins': [0, 200, 500, 1000, 2000, np.inf],
        'labels': ['<200', '200-500', '500-1K', '1K-2K', '>2K']
    },

    'posts_per_week': {
        'bins': [0, 1, 3, 5, 8],
        'labels': ['Low Sharing', 'Moderate Sharing', 'Active Sharing', 'High Sharing']
    }
}

In [ ]:
features_visuals(df_cleaned, engagement_bins)

In [ ]:
features_categorical_socail(social_categorical_features, 'Social Influence')

## comparisions between Age Vs PLatform used

Using Pie

In [ ]:
import plotly.express as px
import pandas as pd

# Get unique cluster names
unique_clusters = df_cleaned['Cluster_Name'].unique()

for cluster_name in unique_clusters:
    # Filter data for the current cluster
    cluster_data = df_cleaned[df_cleaned['Cluster_Name'] == cluster_name]

    # Calculate the distribution of 'primary_platform' within this cluster
    platform_distribution = cluster_data['primary_platform'].value_counts().reset_index()
    platform_distribution.columns = ['primary_platform', 'Count']

    # Create a pie chart for the current cluster
    fig = px.pie(
        platform_distribution,
        values='Count',
        names='primary_platform',
        title=f'Primary Platform Distribution for {cluster_name}',
        width=800,
        height=600,
        color_discrete_sequence=px.colors.qualitative.Set2
    )

    fig.update_traces(
        textposition='inside',
        textinfo='percent+label',
        marker=dict(line=dict(color='#000000', width=1))
    )

    fig.update_layout(
        title_x=0.5,
        uniformtext_minsize=12,
        uniformtext_mode='hide'
    )

    fig.show()


Rader chart


In [ ]:
import plotly.express as px
import pandas as pd

# Group data by Cluster_Name and primary_platform, then calculate the average age
radar_data = df_cleaned.groupby(['Cluster_Name', 'primary_platform'])['age'].mean().reset_index()

# Create the radar chart
fig = px.line_polar(
    radar_data,
    r='age',
    theta='primary_platform',
    color='Cluster_Name',
    line_close=True, # Connect the last point to the first
    title='Average Age by Primary Platform Across User Clusters (Radar Chart)',
    width=800,
    height=600,
    color_discrete_sequence=px.colors.qualitative.Set2 # Consistent color scheme
)

fig.update_traces(fill='toself') # Fill the area under the lines

fig.update_layout(
    title_x=0.5,
    polar={
        'radialaxis': {
            'visible': True,
            'range': [radar_data['age'].min() * 0.9, radar_data['age'].max() * 1.1] # Adjust range for better visualization
        }
    },
    legend_title_text='User Cluster'
)

fig.show()

# Spending & Marketing Clustering

In [ ]:
Spending_marketing_features = [
    'ad_click_rate',
    'purchased_via_social_media',
    'monthly_spend_via_social_usd',
    'primary_purpose'
]
spending_numirical_features = [f for f in Spending_marketing_features if f in numeric_features]
spending_categorical_features = [f for f in Spending_marketing_features if f in categorical_features]

for f in spending_numirical_features:
    print(f)

print("\ncategorical data")
for f in spending_categorical_features:
    print(f)

In [ ]:

engagement_bins_spending = {
    'ad_click_rate': {
        'bins': [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.10, 0.15, 0.20, 0.30, np.inf],
        'labels': ['<1%', '1-2%', '2-3%', '3-4%', '4-5%', '5-10%', '10-15%', '15-20%', '20-30%', '>30%']
    },

    'monthly_spend_via_social_usd': {
        'bins': [0, 10, 20, 30, 50, 100, np.inf],
        'labels': [
            '<10',
            '10-20',
            '20-30',
            '30-50',
            '50-100',
            '>100'
        ]
    }
}


In [ ]:
features_visuals(df_cleaned, engagement_bins_spending)

In [ ]:
crosstab= pd.crosstab(
    df_cleaned['Cluster_Name'],
    df_cleaned['monthly_spend_via_social_usd']
)
print(crosstab)

In [ ]:
features_categorical_socail(spending_categorical_features, 'Spending & Marketing')

# Demographic + Lifestyle Clustering

In [ ]:
demographic_lifestyle_features = np.array([
    'age',
    'gender',
    'country',
    'profession',
    'preferred_device',
    'privacy_setting'
])
demographic_numirical_features = [f for f in demographic_lifestyle_features if f in numeric_features]
demographic_categorical_features = [f for f in demographic_lifestyle_features if f in categorical_features]

for f in demographic_numirical_features:
    print(f)

print("\ncategorical data")
for f in demographic_categorical_features:
    print(f)

age

categorical data
gender
country
profession
preferred_device
privacy_setting


In [ ]:
compare_two_features_with_clusters(df_cleaned,'age', 'monthly_spend_via_social_usd')

In [ ]:
engagement_bins_age = {
    'age': {
        'bins': [12, 18, 21, 24, 28, 30, np.inf],
        'labels': ['12-18','18-21','21-24','24-28','28-30','>30']
    }
}

features_visuals(df_cleaned, engagement_bins_age)

In [ ]:
features_categorical_socail(demographic_categorical_features, 'Demographic & Lifestyle')

# Mental Health & Usage Impact Clustering

In [ ]:
mental_health_features = np.array([
    'sleep_disruption',
    'self_reported_mental_health_score',
    'screen_time_concern',
    'daily_usage_hours'
])

In [ ]:
mental_healt_numirical_features = [f for f in mental_health_features if f in numeric_features]
mental_healt_categorical_features = [f for f in mental_health_features if f in categorical_features]

for f in mental_healt_numirical_features:
    print(f)

print("\ncategorical data")
for f in mental_healt_categorical_features:
    print(f)

self_reported_mental_health_score
daily_usage_hours

categorical data
sleep_disruption
screen_time_concern


In [ ]:
engagement_bins_health = {
    'self_reported_mental_health_score': {
        'bins': [0, 2, 4, 6, 8, 10],
        'labels': [
            'Very Low',
            'Low',
            'Moderate',
            'Good',
            'Excellent'
        ]
    },

    'daily_usage_hours': {
        'bins': [0, 1, 2, 4, 6, 8, 12, np.inf],
        'labels': [
            '<1 hr',
            '1-2 hrs',
            '2-4 hrs',
            '4-6 hrs',
            '6-8 hrs',
            '8-12 hrs',
            '>12 hrs'
        ]
    }
}

features_visuals(df_cleaned, engagement_bins_health)

In [ ]:
features_categorical_socail(mental_healt_categorical_features, 'Mental Health & Usage Impact')

#Platform Usage Behavior

In [ ]:
platform_features = np.array([
    'primary_platform',
    'platforms_used_count',
    'peak_usage_time',
    'preferred_content_type',
    'privacy_setting'
])

In [ ]:
platform_numirical_features = [f for f in platform_features if f in numeric_features]
platform_categorical_features = [f for f in platform_features if f in categorical_features]

for f in platform_numirical_features:
    print(f)

print("\ncategorical data")
for f in platform_categorical_features:
    print(f)

platforms_used_count

categorical data
primary_platform
peak_usage_time
preferred_content_type
privacy_setting


In [ ]:
engagement_bins_health = {
    'platforms_used_count': {
        'bins': [0, 1, 2, 3, 5, np.inf],
        'labels': [
            '1 Platform',
            '2 Platforms',
            '3 Platforms',
            '4-5 Platforms',
            '>5 Platforms'
        ]
    }
}
features_visuals(df_cleaned, engagement_bins_health)

In [ ]:
features_categorical_socail(platform_categorical_features, 'Platform Usage')